In [41]:
# Checa GPU escolhida A100
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Tue Apr  7 17:08:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             33W /   70W |    4885MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
# Conecta-se com o Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [43]:
# Navega para a pasta onde estão os arquivos e programas
%cd /content/drive/My Drive/aval1_qobjetivas

/content/drive/My Drive/aval1_qobjetivas


In [44]:
# Instala pacote zstd necessário para baixar e executar o ollama no Colab
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [45]:
# Instala o binário do Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [46]:
# Instala o Ollama
!pip install ollama

In [47]:
# 1. Para o servidor atual se estiver rodando, em caso de algum erro, se for necessário
# !pkill ollama

# 2. Define que o Ollama pode manter até 3 modelos na memória, pois comporta na GPU A100 da assinatura Colab Pro
# Caso a GPU escolhida não comporte os 3 modelos, usar OLLAMA_MAX_LOADED_MODELS=1 ou comentar, pois é o default
%env OLLAMA_MAX_LOADED_MODELS=1

# 3. Define por quanto tempo eles ficam na memória (ex: 60 minutos ou -1 para sempre)
# %env OLLAMA_KEEP_ALIVE=60m
%env OLLAMA_KEEP_ALIVE=-1

# 4. Inicia o servidor ollama em segundo plano e libera o Colab para a próxima tarefa
# O '&' no final joga o processo para o background e redireciona mensagens para /dev/null
!ollama serve > /dev/null 2>&1 &

env: OLLAMA_MAX_LOADED_MODELS=1
env: OLLAMA_KEEP_ALIVE=-1


In [48]:
# Checar se o ollama está rodando
!curl http://localhost:11434

Ollama is running

In [49]:
import requests
try:
    res = requests.get("http://localhost:11434/api/tags")
    print("Ollama está ON-LINE! Modelos disponíveis:", res.json())
except:
    print("Ollama está OFF-LINE. Rode o 'ollama serve' novamente.")

Ollama está ON-LINE! Modelos disponíveis: {'models': [{'name': 'jurema:latest', 'model': 'jurema:latest', 'modified_at': '2026-04-07T15:52:29.070811619Z', 'size': 4683074535, 'digest': '116807be78ec0c13ba9bfdc52606e0fa38d9c003af5b6aeba3c4fb2069a5dc74', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '7.6B', 'quantization_level': 'Q4_K_M'}}, {'name': 'gemma_safe:latest', 'model': 'gemma_safe:latest', 'modified_at': '2026-04-07T15:48:57.839580297Z', 'size': 8149190672, 'digest': 'd1619e0f97b79a7ef5c04667a901d04a968ba3348234dc9816eb232008aafa07', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '12.2B', 'quantization_level': 'Q4_K_M'}}, {'name': 'llama_safe:latest', 'model': 'llama_safe:latest', 'modified_at': '2026-04-07T15:48:49.431814515Z', 'size': 4920753747, 'digest': '1f08ed660189a317f790842128d17f73d3dc2fc80b39a2e3ba7b6ae31236f869', 'details': {'parent_model'

In [50]:
# Checar versão do ollama
!ollama --version

ollama version is 0.20.3


In [51]:
# Baixa o modelo llama3.1:8b
!ollama pull llama3.1:8b

In [52]:
# Baixa o modelo gemma3:12b
!ollama pull gemma3:12b

In [53]:
# Configura a execução do modelo llama3.1:8b
%%writefile Modelfile_llama_safe
FROM llama3.1:8b
PARAMETER temperature 0
PARAMETER repeat_penalty 1.2
PARAMETER repeat_last_n 128
PARAMETER num_predict 512
# Forçar 100% na GPU
PARAMETER num_gpu 100
#SYSTEM """Você responderá questões objetivas da prova da OAB. Para cada questão, responda apenas com uma única letra maiúscula entre A, B, C e D, correspondente à alternativa correta. Não explique, não justifique, não repita o enunciado e não escreva nada além da letra."""
SYSTEM """Responda questões objetivas da prova da OAB retornando apenas a alternativa correta."""

Overwriting Modelfile_llama_safe


In [54]:
# Cria um alias do modelo llama3.1:8b para ser executado com base na configuração acima
# llama_safe:latest
!ollama create llama_safe -f Modelfile_llama_safe

In [55]:
# Configura a execução do modelo gemma3:12b
%%writefile Modelfile_gemma_safe
FROM gemma3:12b
PARAMETER temperature 0
PARAMETER repeat_penalty 1.2
PARAMETER repeat_last_n 128
PARAMETER num_predict 512
# PARAMETER num_gpu 41
# Forçar 100% na GPU
PARAMETER num_gpu 100
#SYSTEM """Você responderá questões objetivas da prova da OAB. Para cada questão, responda apenas com uma única letra maiúscula entre A, B, C e D, correspondente à alternativa correta. Não explique, não justifique, não repita o enunciado e não escreva nada além da letra."""
SYSTEM """Responda questões objetivas da prova da OAB retornando apenas a alternativa correta."""

Overwriting Modelfile_gemma_safe


In [56]:
# Cria um alias do modelo gemma3:12b para ser executado com base na configuração acima
# gemma_safe:latest
!ollama create gemma_safe -f Modelfile_gemma_safe

In [57]:
# Lista todos os modelos disponíveis, incluindo os novos modelos (latest) configurados para execução
!ollama list

NAME                 ID              SIZE      MODIFIED               
gemma_safe:latest    b71b5c4c7b97    8.1 GB    Less than a second ago    
llama_safe:latest    65b5ce2231f9    4.9 GB    Less than a second ago    
gemma3:12b           f4031aab637d    8.1 GB    Less than a second ago    
llama3.1:8b          46e0c10c039e    4.9 GB    1 second ago              
jurema:latest        116807be78ec    4.7 GB    About an hour ago         


In [58]:
# O modelo Jurema não tem dispoível na biblioteca do Ollama
# O seu arquivo GGUF foi baixado do Hugging Face de disponibilizado no meu drive
# Fazendo uma cópia para do meu Drive para a pasta /content do ambiente Colab, a fim de ficar mais rápido
# do que ler diretamente da minha pasta do Drive durante a execução
!cp /content/drive/MyDrive/aval1_qobjetivas/jurema-7b-q4_k_m.gguf /content/jurema.gguf

In [59]:
# Configura a execução do modelo jurema-7b-q4_k_m.gguf renomeado acima para jurema.gguf
%%writefile Modelfile_jurema
FROM /content/jurema.gguf
#PARAMETER temperature 0
#PARAMETER repeat_penalty 1.1
#PARAMETER repeat_last_n 64
#PARAMETER num_ctx 4096
#PARAMETER num_predict 64

#PARAMETER temperature 0
# 1. Penalidade de repetição (Essencial para temperatura 0)
PARAMETER repeat_penalty 1.2
# 2. Janela de observação de repetição
PARAMETER repeat_last_n 128
# 3. Limite de tokens para evitar respostas infinitas
PARAMETER num_predict 512
# 4. Garante que use a GPU para não levar 6 minutos
# PARAMETER num_gpu 33
# Forçar 100% na GPU
PARAMETER num_gpu 100
SYSTEM """Você responderá questões objetivas da OAB. Quando solicitado, escolha a alternativa correta."""
#SYSTEM """Você responderá questões objetivas da prova da OAB. Para cada questão, responda apenas com uma única letra maiúscula entre A, B, C e D, correspondente à alternativa correta. Não explique, não justifique, não repita o enunciado e não escreva nada além da letra."""

Overwriting Modelfile_jurema


In [60]:
# Cria um alias do modelo jurema.gguf para ser executado com base na configuração acima
!ollama create jurema -f Modelfile_jurema

In [61]:
#!ollama stop jurema:latest
!ollama ps

NAME             ID              SIZE      PROCESSOR    CONTEXT    UNTIL   
jurema:latest    116807be78ec    4.9 GB    100% GPU     4096       Forever    


In [62]:
# Lista novamente todos os modelos disponíveis, incluindo os novos modelos (latest) configurados para execução
# desta feita incluindo o juremma:latest
!ollama list

NAME                 ID              SIZE      MODIFIED               
jurema:latest        116807be78ec    4.7 GB    Less than a second ago    
gemma_safe:latest    b71b5c4c7b97    8.1 GB    About a minute ago        
llama_safe:latest    65b5ce2231f9    4.9 GB    About a minute ago        
gemma3:12b           f4031aab637d    8.1 GB    About a minute ago        
llama3.1:8b          46e0c10c039e    4.9 GB    About a minute ago        


In [63]:
# Testando o modelo jurema:latest
# %%bash
# curl -s http://localhost:11434/api/chat \
#   -H "Content-Type: application/json" \
#   -d '{
#     "model": "jurema:latest",
#     "stream": false,
#     "messages": [
#       {
#         "role": "system",
#         "content": "Responda questões objetivas da prova da OAB retornando apenas a alternativa correta."
#       },
#       {
#         "role": "user",
#         "content": "Como é sabido, o Pacto Internacional dos Direitos Civis e Políticos estabelece em seu Art. 25 que todo cidadão terá o direito e a possibilidade de votar e de ser eleito em eleições periódicas, autênticas, realizadas por sufrágio universal e igualitário e por voto secreto, que garantam a manifestação da vontade dos eleitores. Segundo informação da Agência Brasil (Empresa Brasileira de Comunicação), o Brasil possuía, em 2014, cerca de 230 mil presos provisórios.\nEm relação a tais presos, assinale a afirmativa correta.\n\nA) A despeito do Pacto supramencionado, eles não possuem direito ao voto, por estarem em situação de encarceramento, o que enseja perda da condição de cidadão.\nB) Tais presos provisórios têm direito ao voto apenas se manifestarem expressamente o interesse em votar e forem previamente cadastrados pelo TRE.\nC) Todos aqueles que estão privados de liberdade por ato legal do Estado perdem seus direitos políticos, não podendo, portanto, votar e nem se candidatar.\nD) Presos provisórios têm o direito de votar em seções eleitorais especiais devidamente instaladas em estabelecimentos penais e em unidades de internação de adolescentes."
#       }
#     ],
#     "format": {
#       "type": "object",
#       "properties": {
#         "resposta": {
#           "type": "string",
#           "enum": ["A","B","C","D"]
#         }
#       },
#       "required": ["resposta"]
#     },
#         "options": {
#             "temperature": 0
#         },
#         "stream": false
#   }'

In [64]:
# Testando o modelo llama_safe:latest
# %%bash
# curl -s http://localhost:11434/api/chat \
#   -H "Content-Type: application/json" \
#   -d '{
#     "model": "llama_safe:latest",
#     "stream": false,
#     "messages": [
#       {
#         "role": "system",
#         "content": "Responda questões objetivas da prova da OAB retornando apenas a alternativa correta."
#       },
#       {
#         "role": "user",
#         "content": "Como é sabido, o Pacto Internacional dos Direitos Civis e Políticos estabelece em seu Art. 25 que todo cidadão terá o direito e a possibilidade de votar e de ser eleito em eleições periódicas, autênticas, realizadas por sufrágio universal e igualitário e por voto secreto, que garantam a manifestação da vontade dos eleitores. Segundo informação da Agência Brasil (Empresa Brasileira de Comunicação), o Brasil possuía, em 2014, cerca de 230 mil presos provisórios.\nEm relação a tais presos, assinale a afirmativa correta.\n\nA) A despeito do Pacto supramencionado, eles não possuem direito ao voto, por estarem em situação de encarceramento, o que enseja perda da condição de cidadão.\nB) Tais presos provisórios têm direito ao voto apenas se manifestarem expressamente o interesse em votar e forem previamente cadastrados pelo TRE.\nC) Todos aqueles que estão privados de liberdade por ato legal do Estado perdem seus direitos políticos, não podendo, portanto, votar e nem se candidatar.\nD) Presos provisórios têm o direito de votar em seções eleitorais especiais devidamente instaladas em estabelecimentos penais e em unidades de internação de adolescentes."
#       }
#     ],
#     "format": {
#       "type": "object",
#       "properties": {
#         "resposta": {
#           "type": "string",
#           "enum": ["A","B","C","D"]
#         }
#       },
#       "required": ["resposta"]
#     },
#         "options": {
#             "temperature": 0
#         },
#         "stream": false
#   }'

In [65]:
# Testando o modelo gemma_safe:latest
# %%bash
# curl -s http://localhost:11434/api/chat \
#   -H "Content-Type: application/json" \
#   -d '{
#     "model": "gemma_safe:latest",
#     "stream": false,
#     "messages": [
#       {
#         "role": "system",
#         "content": "Responda questões objetivas da prova da OAB retornando apenas a alternativa correta."
#       },
#       {
#         "role": "user",
#         "content": "Como é sabido, o Pacto Internacional dos Direitos Civis e Políticos estabelece em seu Art. 25 que todo cidadão terá o direito e a possibilidade de votar e de ser eleito em eleições periódicas, autênticas, realizadas por sufrágio universal e igualitário e por voto secreto, que garantam a manifestação da vontade dos eleitores. Segundo informação da Agência Brasil (Empresa Brasileira de Comunicação), o Brasil possuía, em 2014, cerca de 230 mil presos provisórios.\nEm relação a tais presos, assinale a afirmativa correta.\n\nA) A despeito do Pacto supramencionado, eles não possuem direito ao voto, por estarem em situação de encarceramento, o que enseja perda da condição de cidadão.\nB) Tais presos provisórios têm direito ao voto apenas se manifestarem expressamente o interesse em votar e forem previamente cadastrados pelo TRE.\nC) Todos aqueles que estão privados de liberdade por ato legal do Estado perdem seus direitos políticos, não podendo, portanto, votar e nem se candidatar.\nD) Presos provisórios têm o direito de votar em seções eleitorais especiais devidamente instaladas em estabelecimentos penais e em unidades de internação de adolescentes."
#       }
#     ],
#     "format": {
#       "type": "object",
#       "properties": {
#         "resposta": {
#           "type": "string",
#           "enum": ["A","B","C","D"]
#         }
#       },
#       "required": ["resposta"]
#     },
#         "options": {
#             "temperature": 0
#         },
#         "stream": false
#   }'

In [66]:
# Verificando onde os modelos estão sendo executados, se 100% na GPU ou parcialmente CPU/GPU ou completamente CPU
!ollama ps

NAME             ID              SIZE      PROCESSOR    CONTEXT    UNTIL   
jurema:latest    116807be78ec    4.9 GB    100% GPU     4096       Forever    


In [67]:
# Instalação das dependências necessárias para processamento das questões
# objetivas, acurácia e geração dos gráficos
!pip install -q pandas tqdm requests matplotlib tabulate

In [68]:
# Rodar programa 01
# Submete as 123 questões objetivas aos 3 modelos e calcula a acurácia
!python3 program_01_objq.py

temperatura 0

>>> Iniciando processamento do modelo: llama_safe:latest
Status llama_safe:latest: 100% 123/123 [02:45<00:00,  1.34s/it]

>>> Iniciando processamento do modelo: gemma_safe:latest
Status gemma_safe:latest: 100% 123/123 [04:56<00:00,  2.41s/it]

>>> Iniciando processamento do modelo: jurema:latest
Status jurema:latest: 100% 123/123 [02:35<00:00,  1.27s/it]

Fim do program_01. Resultados salvos em /content/drive/MyDrive/aval1_qobjetivas/avaliacao_objetivas_paulo.csv


In [98]:
# Gerar os gráficos
!python3 program_02_objq.py

Figure(800x600)
Figure(900x600)
Figure(1400x800)
Figure(900x600)
Figure(700x600)


In [70]:
#!ollama stop llama_safe:latest
#!ollama stop gemma_safe:latest
#!ollama stop jurema:latest
